# Early-Epoch Long-Form Settings Sweep

This notebook keeps the same early checkpoint and the same random songs fixed, then sweeps a thoughtful panel of long-form coherence settings.

Goal:
- hear what the model itself sounds like at an early checkpoint
- separate checkpoint quality from long-form dial effects
- compare 12 settings that trade off style strength, source anchoring, and crackle resistance

Checkpoint selection rule:
- prefer `epoch_003.pt`
- otherwise use `epoch_002.pt`
- otherwise fall back to `epoch_001.pt`

The same songs and same start offsets are reused across all settings.

In [ ]:
from pathlib import Path
import importlib
import json
import sys

import pandas as pd

def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'dggr').exists() and (path / 'lab 4').exists() and (path / 'lab 3.1').exists():
            return path
    raise RuntimeError('Could not resolve repo root from current working directory.')

REPO = find_repo_root(Path.cwd().resolve())
SCRIPTS = REPO / 'lab 3.1' / 'scripts'
if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

import diffusion_longform_settings_sweep as sweep
importlib.reload(sweep)
REPO

In [ ]:
def latest_finished_early_epoch(root: Path) -> tuple[Path, Path]:
    runs = []
    for run_dir in root.iterdir():
        if not run_dir.is_dir():
            continue
        ckpt_dir = run_dir / 'checkpoints'
        for name in ['epoch_003.pt', 'epoch_002.pt', 'epoch_001.pt']:
            ckpt = ckpt_dir / name
            if ckpt.exists():
                runs.append((run_dir, ckpt))
                break
    if not runs:
        raise RuntimeError('No epoch_003/002/001 checkpoints found under diffusion_vocal_crackle_retool.')
    runs.sort(key=lambda item: item[1].stat().st_mtime, reverse=True)
    return runs[0]

RUN_DIR, CHECKPOINT_PATH = latest_finished_early_epoch(REPO / 'lab 3.1' / 'outputs' / 'diffusion_vocal_crackle_retool')

cfg = sweep.DiffusionSettingsSweepConfig(
    downloads_dir=Path.home() / 'Downloads',
    run_dir=RUN_DIR,
    checkpoint_path=CHECKPOINT_PATH,
    cache_dir=REPO / 'saves2' / 'lab3_diffusion' / 'run_d001' / 'cache',
    n_songs=2,
    targets_per_song=2,
    source_seconds=45.0,
    seed=328,
)

RUN_ALL =   True

print('Run dir:     ', RUN_DIR)
print('Checkpoint:  ', CHECKPOINT_PATH)
print('Output root: ', cfg.output_root / cfg.tag)

In [ ]:
settings_panel = sweep.default_settings_panel()
display(pd.DataFrame(settings_panel))
print('Total settings:', len(settings_panel))

In [ ]:
plan_cfg = sweep.dlc.DiffusionLongformCompareConfig(
    downloads_dir=cfg.downloads_dir,
    output_root=cfg.output_root,
    run_dir=cfg.run_dir,
    cache_dir=cfg.cache_dir,
    lab1_checkpoint=cfg.lab1_checkpoint,
    n_songs=cfg.n_songs,
    targets_per_song=cfg.targets_per_song,
    source_seconds=cfg.source_seconds,
    chunk_seconds=cfg.chunk_seconds,
    overlap_seconds=cfg.overlap_seconds,
    n_frames=cfg.n_frames,
    ddim_steps=cfg.ddim_steps,
    assemble_domain=cfg.assemble_domain,
    device=cfg.device,
    seed=cfg.seed,
    snapshot_latest_checkpoint=False,
)
jobs = sweep.dlc.plan_longform_jobs(plan_cfg)
display(pd.DataFrame(jobs))
print('Total jobs:', len(jobs))
print('Total runs:', len(jobs) * len(settings_panel))

In [ ]:
summary = None
if RUN_ALL:
    summary = sweep.run_settings_sweep(cfg, settings_panel)
    print(json.dumps(summary, indent=2, default=str))
else:
    print('Set RUN_ALL = True to launch the epoch-1 settings sweep.')

In [ ]:
summary_path = cfg.output_root / cfg.tag / 'summary.json'
manifest_path = cfg.output_root / cfg.tag / 'manifest.csv'
if summary_path.exists():
    print(summary_path)
    print(manifest_path)
    display(pd.read_csv(manifest_path).head(20))
else:
    print('No outputs yet for this tag.')